In [1]:
from datasets import load_dataset
from transformers import MT5Tokenizer, MT5ForConditionalGeneration
from tqdm import tqdm
import pandas as pd
import torch
import os

# Kiểm tra GPU
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu" )
print(f"💻 Đang sử dụng thiết bị: {device}")

# Load model paraphrase và đưa lên GPU
CKPT = 'chieunq/vietnamese-sentence-paraphase'
tokenizer = MT5Tokenizer.from_pretrained(CKPT)
model = MT5ForConditionalGeneration.from_pretrained(CKPT).to(device)

# Hàm tạo paraphrase
def paraphrase(text, num_return_sequences=5):
    inputs = tokenizer(text, padding='longest', max_length=512, truncation=True, return_tensors='pt')
    inputs = {key: val.to(device) for key, val in inputs.items()}
    output = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=512,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_p=0.95
    )
    return [tokenizer.decode(o, skip_special_tokens=True) for o in output]

# Load dataset
dataset = pd.read_parquet("../generate_dataset/SVYKHOA_dataset_guide.parquet", engine="fastparquet")

# File Excel đầu ra
output_file = "SVYKHOA_dataset_guide_2.xlsx"

# Tạo file Excel rỗng nếu chưa có
if not os.path.exists(output_file):
    df_empty = pd.DataFrame(columns=["intruction", "question", "answer","document/title","document/description","cme/title","cme/description"])
    df_empty.to_excel(output_file, index=False)

# Đọc số dòng đã có để tiếp tục từ đó
existing_df = pd.read_excel(output_file)
# start_index = len(existing_df)
start_index = 6000
print(f"🚀 Bắt đầu từ dòng {start_index}")

# Số lượng mẫu muốn xử lý thêm
max_samples = 139490 - 60000  # Có thể chỉnh: 10, 100, 500...

# Duyệt dataset từ start_index
for i, row in dataset.iterrows():
    if i < start_index:
        continue
    if i >= start_index + max_samples:
        break
    print(row)

    question = row["question"]   # row là Series
    try:
        paraphrases = paraphrase(question, num_return_sequences=15)
    except Exception as e:
        print(f"Lỗi paraphrase tại mẫu {i}: {e}")
        paraphrases = [question]

    new_rows = []
    for pq in paraphrases:
        new_rows.append({
            "intruction": row["intruction"],
            "question": pq,
            "answer": row["answer"],
            "document/title": row["document/title"],
            "document/description": row["document/description"],
            "cme/title": row["cme/title"],
            "cme/description": row["cme/description"],
        })

    new_df = pd.DataFrame(new_rows)
    with pd.ExcelWriter(output_file, mode="a", engine="openpyxl", if_sheet_exists="overlay") as writer:
        sheet = writer.sheets["Sheet1"]
        start_row = sheet.max_row
        new_df.to_excel(writer, header=False, index=False, startrow=start_row)


print("✅ Đã lưu toàn bộ dữ liệu paraphrase vào file Excel.")

💻 Đang sử dụng thiết bị: cpu


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.
You are using the default legacy behaviour of the <class 'transformers.models.mt5.tokenization_mt5.MT5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


🚀 Bắt đầu từ dòng 6000
intruction              Chatbot y khoa chuyên về y khoa, tư vấn cho bá...
question                Ngoài liệu pháp âm nhạc, bác sĩ cần lưu ý gì k...
answer                  Chào bạn, việc đánh giá và quản lý lo lắng tiề...
document/title          Cẩm nang đánh giá và quản lý lo âu tiền phẫu t...
document/description    Tài liệu này cung cấp hướng dẫn chi tiết về cá...
cme/title               Kỹ năng giao tiếp và tâm lý học lâm sàng trong...
cme/description         Khóa học này nhằm trang bị cho các bác sĩ và n...
Name: 6000, dtype: object
intruction              Chatbot y khoa chuyên về y khoa, hỗ trợ sinh v...
question                Trong nghiên cứu này, các chỉ số p-value (< 0....
answer                  Chào bạn, việc hiểu và áp dụng kết quả nghiên ...
document/title          Phương pháp nghiên cứu khoa học và thống kê y ...
document/description    Sách giáo trình này cung cấp kiến thức nền tản...
cme/title                     Y học thực chứng và ra quyết định

KeyboardInterrupt: 